In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
import sklearn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import mlflow, mlflow.sklearn
from mlflow.models import infer_signature


In [2]:
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)

In [3]:
data = load_breast_cancer(as_frame=True)
X = data.data          # 569 rows x 30 numeric features (DataFrame)
y = data.target        # 0 = malignant, 1 = benign (Series)

df = data.frame        # features and 'target' together
print(df.shape, y.value_counts().to_dict())

(569, 31) {1: 357, 0: 212}


In [4]:
from sklearn.model_selection import train_test_split

# 1. import the data
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target          # 0 = malignant, 1 = benign

# 2. split 80/20 (stratified so both parts keep the class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=82)

# 3. scale: learn mean/std from the TRAINING rows only, then apply to both parts
scaler = StandardScaler().fit(X_train)
X_train_s = pd.DataFrame(scaler.transform(X_train), columns=X.columns, index=X_train.index)
X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

print("train / test:", X_train_s.shape, X_test_s.shape)

train / test: (512, 30) (57, 30)


In [5]:
X_train.head().T

,18,407,529,467,7
mean radius,19.810000,12.850000,12.070000,9.668000,13.710000
mean texture,22.150000,21.370000,13.440000,18.100000,20.830000
mean perimeter,130.000000,82.630000,77.830000,61.060000,90.200000
mean area,1260.000000,514.500000,445.200000,286.300000,577.900000
mean smoothness,0.098310,0.075510,0.110000,0.083110,0.118900
mean compactness,0.102700,0.083160,0.090090,0.054280,0.164500
mean concavity,0.147900,0.061260,0.037810,0.014790,0.093660
mean concave points,0.094980,0.018670,0.027980,0.005769,0.059850
mean symmetry,0.158200,0.158000,0.165700,0.168000,0.219600
mean fractal dimension,0.053950,0.061140,0.066080,0.064120,0.074510


In [6]:
X_tr_in, X_val_in, y_tr_in, y_val_in = train_test_split(
    X_train_s, y_train, test_size=0.2, stratify=y_train, random_state=82)

print("train / Validation / test:", X_tr_in.shape, X_val_in.shape, X_test_s.shape)

train / Validation / test: (409, 30) (103, 30) (57, 30)


In [7]:
def score(prefix, model, X_, y_):
    pred, proba = model.predict(X_), model.predict_proba(X_)[:, 1]
    return {f"{prefix}_accuracy":  accuracy_score(y_, pred),
            f"{prefix}_precision": precision_score(y_, pred),
            f"{prefix}_recall":    recall_score(y_, pred),
            f"{prefix}_f1":        f1_score(y_, pred),
            f"{prefix}_roc_auc":   roc_auc_score(y_, proba)}

In [8]:
experiment_name = "breast_cancer_models_v1"

mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='/home/cloud_user/Study/harvard/cs082/Assignments/Tool/mlruns/2', creation_time=1793330895339, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1793330895339, lifecycle_stage='active', name='breast_cancer_models_v1', tags={}, trace_location=None, workspace='default'>

In [9]:
exp = mlflow.get_experiment_by_name("breast_cancer_models_v1")
run_ids = mlflow.search_runs(experiment_ids=[exp.experiment_id])["run_id"]

for run_id in run_ids:
    mlflow.delete_run(run_id)

print(f"deleted {len(run_ids)} runs")

deleted 10 runs


### Model 1 - Logistic Regression

In [10]:
lr_settings = {"LR_C=0.01": 0.01, "LR_C=1": 1.0, "LR_C=100": 100.0}

In [11]:
for run_name, C in lr_settings.items():
    params = {"C": C, "solver": "lbfgs", "max_iter": 1000, "random_state": 82}
    model = LogisticRegression(**params)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.set_tags({"model_family": "logistic_regression", "model": "Model 1"})

        model.fit(X_tr_in, y_tr_in)

        metrics = {**score("train", model, X_tr_in, y_tr_in),
                   **score("val",   model, X_val_in, y_val_in),
                   **score("test",  model, X_test_s, y_test)}
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, name="model",
                                signature=infer_signature(X_tr_in, model.predict(X_tr_in)),
                                input_example=X_tr_in.head(5))
        mlflow.sklearn.log_model(scaler, name="scaler")

In [12]:
runs = mlflow.search_runs(
    experiment_names=["breast_cancer_models_v1"],
    filter_string="tags.model_family = 'logistic_regression' AND metrics.val_roc_auc > 0.98",
    order_by=["metrics.val_roc_auc DESC"])

In [13]:
runs.T

,0,1,2
run_id,a4dc15ebfec74e1cb23ef3bed4d31f75,d03f923d3ae24763ba421d876800ee00,d0efc78899e44759839f07685fa21903
experiment_id,2,2,2
status,FINISHED,FINISHED,FINISHED
artifact_uri,/home/cloud_user/Study/harvard/cs082/Assignmen...,/home/cloud_user/Study/harvard/cs082/Assignmen...,/home/cloud_user/Study/harvard/cs082/Assignmen...
start_time,2026-10-31 17:36:38.594000+00:00,2026-10-31 17:37:00.451000+00:00,2026-10-31 17:36:09.268000+00:00
end_time,2026-10-31 17:37:00.438000+00:00,2026-10-31 17:37:15.444000+00:00,2026-10-31 17:36:38.578000+00:00
metrics.train_accuracy,0.99022,0.997555,0.948655
metrics.train_roc_auc,0.998009,0.999974,0.994383
metrics.test_roc_auc,0.996032,0.952381,1.0
metrics.test_accuracy,0.947368,0.947368,0.947368


In [14]:
runs[["tags.model_family", "tags.mlflow.runName", "params.C", "metrics.train_roc_auc", "metrics.val_roc_auc",
      "metrics.test_roc_auc", "metrics.test_accuracy", "metrics.test_f1"]].round(4)


,tags.model_family,tags.mlflow.runName,params.C,metrics.train_roc_auc,metrics.val_roc_auc,metrics.test_roc_auc,metrics.test_accuracy,metrics.test_f1
0,logistic_regression,LR_C=1,1.0,0.9980,0.9947,0.9960,0.9474,0.9589
1,logistic_regression,LR_C=100,100.0,1.0000,0.9935,0.9524,0.9474,0.9577
2,logistic_regression,LR_C=0.01,0.01,0.9944,0.9899,1.0000,0.9474,0.9600


In [15]:
# ! mlflow ui

#### Model 2 - Random Forest

In [16]:
from sklearn.ensemble import RandomForestClassifier
n_features = X_tr_in.shape[1]                      # 30
max_features_options = {"1/3": 1/3, "sqrt": "sqrt"}

for n_estimators in [10, 20]:
    for mf_label, mf in max_features_options.items():
        run_name = f"RF_n={n_estimators}_mf={mf_label}"
        params = {"n_estimators": n_estimators, "max_features": mf_label,
                  "max_features_used": max(1, int(mf * n_features)) if mf != "sqrt" else int(np.sqrt(n_features)),
                  "random_state": 82}
        model = RandomForestClassifier(n_estimators=n_estimators, max_features=mf,
                                       random_state=82, n_jobs=-1)

        with mlflow.start_run(run_name=run_name):
            mlflow.log_params(params)
            mlflow.set_tags({"model_family": "random_forest", "model": "Model 2"})

            model.fit(X_tr_in, y_tr_in)

            metrics = {**score("train", model, X_tr_in, y_tr_in),
                       **score("val",   model, X_val_in, y_val_in),
                       **score("test",  model, X_test_s, y_test)}
            mlflow.log_metrics(metrics)

            mlflow.sklearn.log_model(model, name="model",
                                     signature=infer_signature(X_tr_in, model.predict(X_tr_in)),
                                     input_example=X_tr_in.head(5),
                                     skops_trusted_types=["sklearn.tree._tree.Tree"])  # trees we trained ourselves
            mlflow.sklearn.log_model(scaler, name="scaler")   

In [17]:
runs = mlflow.search_runs(experiment_names=["breast_cancer_models_v1"],  
                          order_by=["metrics.val_roc_auc DESC"])
runs[["run_id", "tags.model_family", "tags.mlflow.runName", "metrics.train_roc_auc", "metrics.val_roc_auc",
      "metrics.test_roc_auc", "metrics.test_accuracy", "metrics.test_f1"]].round(4)

,run_id,tags.model_family,tags.mlflow.runName,metrics.train_roc_auc,metrics.val_roc_auc,metrics.test_roc_auc,metrics.test_accuracy,metrics.test_f1
0,a4dc15ebfec74e1cb23ef3bed4d31f75,logistic_regression,LR_C=1,0.9980,0.9947,0.9960,0.9474,0.9589
1,d03f923d3ae24763ba421d876800ee00,logistic_regression,LR_C=100,1.0000,0.9935,0.9524,0.9474,0.9577
2,d0efc78899e44759839f07685fa21903,logistic_regression,LR_C=0.01,0.9944,0.9899,1.0000,0.9474,0.9600
3,0351853507a040f5909881086795bd6a,random_forest,RF_n=20_mf=sqrt,1.0000,0.9877,0.9914,0.9474,0.9577
4,946b5268fbfc40d0af40b208b81448dd,random_forest,RF_n=20_mf=1/3,1.0000,0.9850,0.9927,0.9825,0.9859
5,56a2d9735eeb47e7b7653f5e92e70bee,random_forest,RF_n=10_mf=sqrt,1.0000,0.9830,0.9921,0.9474,0.9589
6,a16169846e5f43899e83f8ee38aa2700,random_forest,RF_n=10_mf=1/3,1.0000,0.9678,0.9921,0.9825,0.9859


### Model 3 Nueral network

In [18]:
import tempfile, logging, warnings
import torch
import torch.nn as nn
import lightning.pytorch as pl
import mlflow.pytorch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import MLFlowLogger
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import BinaryAUROC


class Net3(pl.LightningModule):
    """3 linear layers: 30 -> 32 -> 16 -> 1 (the last one outputs a logit)."""
    def __init__(self, n_features=30, h1=32, h2=16, lr=1e-3, weight_decay=1e-4):
        super().__init__()
        self.save_hyperparameters()
        self.net = nn.Sequential(
            nn.Linear(n_features, h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.ReLU(),
            nn.Linear(h2, 1))
        self.loss_fn = nn.BCEWithLogitsLoss()
        self.val_auc = BinaryAUROC()

    def forward(self, x):
        return self.net(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        loss = self.loss_fn(self(x), y)
        self.log("train_loss", loss, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        self.log("val_loss", self.loss_fn(logits, y), on_epoch=True)
        self.val_auc.update(torch.sigmoid(logits), y.int())

    def on_validation_epoch_end(self):
        self.log("val_auc", self.val_auc.compute())
        self.val_auc.reset()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)


def to_loader(X_, y_, shuffle):
    ds = TensorDataset(torch.tensor(X_.values, dtype=torch.float32),
                       torch.tensor(np.asarray(y_), dtype=torch.float32))
    return DataLoader(ds, batch_size=32, shuffle=shuffle)

def predict_proba_nn(net, X_):
    net.eval()
    with torch.no_grad():
        return torch.sigmoid(net(torch.tensor(X_.values, dtype=torch.float32)).squeeze(1)).numpy()

def score_from_proba(prefix, y_, proba):
    pred = (proba >= 0.5).astype(int)
    return {f"{prefix}_accuracy":  accuracy_score(y_, pred),
            f"{prefix}_precision": precision_score(y_, pred),
            f"{prefix}_recall":    recall_score(y_, pred),
            f"{prefix}_f1":        f1_score(y_, pred),
            f"{prefix}_roc_auc":   roc_auc_score(y_, proba)}


train_dl = to_loader(X_tr_in, y_tr_in, shuffle=True)
val_dl = to_loader(X_val_in, y_val_in, shuffle=False)

for lr in [1e-2, 2e-3, 1e-3]:
    run_name = f"NN_lr={lr}"
    pl.seed_everything(82, workers=True)
    model = Net3(n_features=X_tr_in.shape[1], h1=32, h2=16, lr=lr, weight_decay=1e-4)

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({"layers": "30-32-16-1", "activation": "relu", "lr": lr,
                           "weight_decay": 1e-4, "batch_size": 32, "optimizer": "Adam",
                           "max_epochs": 200, "early_stopping_patience": 15, "random_state": 82})
        mlflow.set_tags({"model_family": "neural_network", "model": "Model 3"})

        with tempfile.TemporaryDirectory() as tmp:
            ckpt = ModelCheckpoint(dirpath=tmp, monitor="val_loss", mode="min", save_top_k=1)
            trainer = pl.Trainer(
                max_epochs=200,
                callbacks=[EarlyStopping(monitor="val_loss", mode="min", patience=15), ckpt],
                logger=MLFlowLogger(tracking_uri=mlflow.get_tracking_uri(), run_id=run.info.run_id),
                enable_progress_bar=False, enable_model_summary=False, accelerator="auto", devices=1)
            trainer.fit(model, train_dl, val_dl)
            best = Net3.load_from_checkpoint(ckpt.best_model_path, map_location="cpu")       # best epoch, not the last
            best_epoch = int(ckpt.best_model_path.split("epoch=")[1].split("-")[0])

        metrics = {**score_from_proba("train", y_tr_in,  predict_proba_nn(best.net, X_tr_in)),
                   **score_from_proba("val",   y_val_in, predict_proba_nn(best.net, X_val_in)),
                   **score_from_proba("test",  y_test,   predict_proba_nn(best.net, X_test_s)),
                   "best_epoch": best_epoch}
        mlflow.log_metrics(metrics)

        example = X_tr_in.values[:5].astype("float32")      # the PyTorch flavor wants numpy, not a DataFrame
        best.net.eval()
        with torch.no_grad():
            out = best.net(torch.tensor(example)).numpy()
        mlflow.pytorch.log_model(best.net, name="model",
                                 signature=infer_signature(example, out),
                                 input_example=example,
                                 serialization_format="pickle")      # loads back as a normal nn.Module
        mlflow.sklearn.log_model(scaler, name="scaler")

Seed set to 82
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Seed set to 82
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Seed set

In [19]:
runs = mlflow.search_runs(experiment_names=["breast_cancer_models_v1"],   # your experiment name
                          order_by=["metrics.val_roc_auc DESC"])
runs[["tags.model_family", "tags.mlflow.runName", "metrics.train_roc_auc", "metrics.val_roc_auc",
      "metrics.test_roc_auc", "metrics.test_accuracy", "metrics.test_f1"]].round(4)

,tags.model_family,tags.mlflow.runName,metrics.train_roc_auc,metrics.val_roc_auc,metrics.test_roc_auc,metrics.test_accuracy,metrics.test_f1
0,neural_network,NN_lr=0.01,1.0000,0.9992,0.9735,0.9298,0.9444
1,neural_network,NN_lr=0.001,1.0000,0.9951,0.9550,0.9298,0.9444
2,neural_network,NN_lr=0.002,0.9999,0.9947,0.9643,0.9298,0.9444
3,logistic_regression,LR_C=1,0.9980,0.9947,0.9960,0.9474,0.9589
4,logistic_regression,LR_C=100,1.0000,0.9935,0.9524,0.9474,0.9577
5,logistic_regression,LR_C=0.01,0.9944,0.9899,1.0000,0.9474,0.9600
6,random_forest,RF_n=20_mf=sqrt,1.0000,0.9877,0.9914,0.9474,0.9577
7,random_forest,RF_n=20_mf=1/3,1.0000,0.9850,0.9927,0.9825,0.9859
8,random_forest,RF_n=10_mf=sqrt,1.0000,0.9830,0.9921,0.9474,0.9589
9,random_forest,RF_n=10_mf=1/3,1.0000,0.9678,0.9921,0.9825,0.9859


### Loading the models

In [20]:
run_id = runs.iloc[6]["run_id"]
model_final  = mlflow.sklearn.load_model(f"runs:/{run_id}/model")
scaler = mlflow.sklearn.load_model(f"runs:/{run_id}/scaler")
# # pred = model.predict(scaler.transform(X_test))

In [21]:
# ! mlflow ui

In [22]:
model_final

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",82
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total 

In [23]:
model_final.feature_importances_

array([0.00796982, 0.0113651 , 0.09975529, 0.03810187, 0.00856653,
       0.01786631, 0.01193137, 0.11014678, 0.00177387, 0.001813  ,
       0.00797657, 0.00277022, 0.008208  , 0.01301506, 0.00338725,
       0.00591583, 0.02131384, 0.00368267, 0.00284149, 0.0037492 ,
       0.10518133, 0.01423975, 0.14609523, 0.10320703, 0.01185275,
       0.02946552, 0.04539665, 0.14685971, 0.01005044, 0.00550154])

In [26]:
pd.DataFrame([model_final.feature_importances_], columns=model_final.feature_names_in_).T
# imp = (pd.Series(model_final.feature_importances_, index=model_final.feature_names_in_)
#          .sort_values(ascending=False)
#          .rename("importance")            # name of the values (column header)
#          .rename_axis("feature"))  

,0
mean radius,0.007970
mean texture,0.011365
mean perimeter,0.099755
mean area,0.038102
mean smoothness,0.008567
mean compactness,0.017866
mean concavity,0.011931
mean concave points,0.110147
mean symmetry,0.001774
mean fractal dimension,0.001813
